In [0]:
df_laptimes = spark.read.csv("/Volumes/gr5069/raw/f1_data/lap_times.csv",header=True)

In [0]:
display(df_laptimes)

raceId,driverId,lap,position,time,milliseconds
841,20,1,1,1:38.109,98109
841,20,2,1,1:33.006,93006
841,20,3,1,1:32.713,92713
841,20,4,1,1:32.803,92803
841,20,5,1,1:32.342,92342
841,20,6,1,1:32.605,92605
841,20,7,1,1:32.502,92502
841,20,8,1,1:32.537,92537
841,20,9,1,1:33.240,93240
841,20,10,1,1:32.572,92572


In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv",header=True)

In [0]:
display(df_drivers)

driverId,driverRef,number,code,forename,surname,dob,nationality,url
1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton
2,heidfeld,\N,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld
3,rosberg,6,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg
4,alonso,14,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso
5,kovalainen,\N,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen
6,nakajima,\N,NAK,Kazuki,Nakajima,1985-01-11,Japanese,http://en.wikipedia.org/wiki/Kazuki_Nakajima
7,bourdais,\N,BOU,Sébastien,Bourdais,1979-02-28,French,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bourdais
8,raikkonen,7,RAI,Kimi,Räikkönen,1979-10-17,Finnish,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C3%B6nen
9,kubica,88,KUB,Robert,Kubica,1984-12-07,Polish,http://en.wikipedia.org/wiki/Robert_Kubica
10,glock,\N,GLO,Timo,Glock,1982-03-18,German,http://en.wikipedia.org/wiki/Timo_Glock


In [0]:
#1. [10 pts] What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

from pyspark.sql import functions as F

# 1. Load the data 
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True, inferSchema=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)

# 2. Join and cast duration to float for calculation
df_joined = df_pitstops.join(df_races, "raceId") \
    .withColumn("duration_float", F.when(F.col("duration").contains(":"), 
        F.regexp_extract(F.col("duration"), r"^(\d+):", 1).cast("float") * 60 + 
        F.regexp_extract(F.col("duration"), r":(.+)$", 1).cast("float"))
        .otherwise(F.col("duration").cast("float")))


# 3. Aggregate: Average per driver per race, plus Race-level extremes
df_driver_avg = df_joined.groupBy("raceId", "name", "driverId") \
    .agg(F.avg("duration_float").alias("avg_driver_pit_time"))

# 4. Now, get the slowest and fastest pit stop for EACH RACE overall
df_race_stats = df_joined.groupBy("raceId", "name") \
    .agg(
        F.avg("duration_float").alias("race_avg_time"),
        F.min("duration_float").alias("fastest_pit_stop"),
        F.max("duration_float").alias("slowest_pit_stop")
    )

# Show results
df_race_stats.orderBy("raceId").show()

+------+--------------------+------------------+------------------+------------------+
|raceId|                name|     race_avg_time|  fastest_pit_stop|  slowest_pit_stop|
+------+--------------------+------------------+------------------+------------------+
|   841|Australian Grand ...|24.342822096082898|16.867000579833984| 37.85599899291992|
|   842|Malaysian Grand Prix|24.434372918080474| 21.89299964904785|38.823001861572266|
|   843|  Chinese Grand Prix|22.299241329061573|13.899999618530273| 33.82699966430664|
|   844|  Turkish Grand Prix|  23.0050609518842|13.925000190734863|56.611000061035156|
|   845|  Spanish Grand Prix| 21.56759735206505|19.534000396728516| 29.93600082397461|
|   846|   Monaco Grand Prix|27.548348759495934|19.347000122070312| 47.22600173950195|
|   847| Canadian Grand Prix| 25.26733332316081| 14.50100040435791|51.683998107910156|
|   848| European Grand Prix|21.795815423818734| 20.13599967956543|30.665000915527344|
|   849|  British Grand Prix| 25.9642222369

To accurately determine the average pit stop time per race—as well as the fastest and slowest times for each event—I devised the following logical workflow: Data Integration and Alignment: First, I perform an INNER JOIN between the `pit_stops` table and the `races` table using the common column `raceId`. This links the specific pit stop times to their corresponding race names (e.g., "Monaco Grand Prix"). Time Format Conversion: Since F1 pit stop times can span across minutes (formatted as M:SS.ms), a direct numerical conversion would result in an error. Logically, I check whether the time string contains a colon: if a colon is present, I split the string into minutes and seconds to calculate the total duration in seconds ($minutes \times 60 + seconds$); if no colon is present, I convert the value directly into a floating-point number. Two-Tier Aggregation Analysis: Driver Level: I group the data by race and driver ID to calculate the average pit stop time, thereby enabling an assessment of individual driver performance. Race Level: I group the data by race to calculate the average, minimum (fastest), and maximum (slowest) pit stop times across all participants in that event. Sorted Output: Finally, I sort the results by `raceId` to ensure the output is presented in chronological or logical order.

Code EXplain
`F.when(...).otherwise(...)` implements conditional logic similar to `if-else`. In data cleaning, it's used to distinguish between processing "minute-based time" and "pure second-based time".

`F.split(..., ":")` splits a string into an array by a colon. For example, "1:15.5" becomes ["1", "15.5"].

`.getItem(0)` extracts the first element (the number of minutes) from the array generated by `split`.

`F.avg()`, `F.min()`, and `F.max()` are standard aggregate functions. `avg` calculates the average performance across different levels, while `min` and `max` correspond to the "fastest" and "slowest" requirements, respectively.

`alias()` renames the calculated result column (e.g., `fastest_pit_stop`) to improve the readability of the result table.

In [0]:
# 2. [20 pts] Rank order by finishing position the average time spent at the pit stop in each race.

from pyspark.sql import functions as F

# 1. load extra data results
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)

# 2. Same step like Q1
df_pit_cleaned = df_pitstops.withColumn("duration_float", 
    F.when(F.col("duration").contains(":"), 
        F.split(F.col("duration"), ":").getItem(0).cast("float") * 60 + 
        F.split(F.col("duration"), ":").getItem(1).cast("float")
    ).otherwise(F.col("duration").cast("float"))
)

# 3. Linking inbound data with match results
df_merged = df_pit_cleaned.join(df_results, ["raceId", "driverId"])

# 4. Calculate average pit stop time based on final race standings.
df_rank_stats = df_merged.groupBy("positionOrder") \
    .agg(F.avg("duration_float").alias("avg_pit_time")) \
    .orderBy("positionOrder")

# Result
df_rank_stats.show(20)

+-------------+------------------+
|positionOrder|      avg_pit_time|
+-------------+------------------+
|            1| 96.84776267430921|
|            2| 96.87015910691852|
|            3| 94.97298596822944|
|            4| 93.49867871946486|
|            5| 94.72191797397107|
|            6| 96.38367076191614|
|            7| 96.48803556791208|
|            8| 97.51133634470686|
|            9| 95.32660666212335|
|           10| 95.65997371569014|
|           11| 94.42360136824375|
|           12| 92.64655215551556|
|           13| 86.37388112801294|
|           14| 83.79302752647295|
|           15| 81.29314217095177|
|           16| 83.75528031802568|
|           17| 75.65054967958633|
|           18| 55.29939484028589|
|           19| 44.52088828607863|
|           20|35.906795610476586|
+-------------+------------------+
only showing top 20 rows


To investigate the relationship between "pit stop duration" and "final ranking," I employed the following logical steps:
First, I joined the `pit_stops` table with the `races` table to retrieve the corresponding race names.
Next, I joined the resulting dataset with the `results` table using the composite primary key consisting of `raceId` and `driverId`, as a single driver's ranking varies across different races.
Adopting the defensive logic from the previous problem, I converted the `duration` strings (originally in the M:SS.ms format) into floating-point seconds. I then grouped the data by `positionOrder` (finishing position) to calculate the average pit stop duration for drivers finishing in each specific rank.
Sorting (Ranking): Finally, I sorted the results in ascending order based on `positionOrder` to allow for a clear visual comparison of pit stop duration differences ranging from the 1st-place finisher (the champion) down to the last-place finisher.

Code Explain
["raceId", "driverId"]	Passing a list to the `join` operation instructs PySpark to use both columns simultaneously as matching conditions, thereby ensuring that we correctly link each pit stop record to the corresponding driver's finishing position in that specific race.

positionOrder	Referenced from the `results` table. Unlike the `position` column (which may contain `null` values ​​or "R" to indicate a retirement), `positionOrder` consists of strictly numerical and unique rankings, making it ideal for sorting purposes.

F.avg("duration_float")	Calculates the arithmetic mean of the pit stop durations for all drivers who achieved a specific finishing position.

.orderBy("positionOrder")	Sorts the results in ascending order based on finishing position (from 1st to 20th), allowing us to quickly observe whether the drivers who reached the podium demonstrated greater pit stop efficiency.

In [0]:
#3. [20 pts] Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

from pyspark.sql import functions as F

# 1. Load drivers data
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True, inferSchema=True)

# 2. 
df_pit_cleaned = df_pitstops.withColumn("duration_float", 
    F.when(F.col("duration").contains(":"), 
        F.split(F.col("duration"), ":").getItem(0).cast("float") * 60 + 
        F.split(F.col("duration"), ":").getItem(1).cast("float")
    ).otherwise(F.col("duration").cast("float"))
)

# 3. Linking inbound data with match results
df_full_merged = df_pit_cleaned \
    .join(df_results, ["raceId", "driverId"]) \
    .join(df_drivers, "driverId")

# 4. Calculate average pit stop time based on final race standings
df_driver_rank_stats = df_full_merged.groupBy("positionOrder", "code") \
    .agg(F.avg("duration_float").alias("avg_pit_time")) \
    .orderBy("positionOrder", "avg_pit_time")

# Result
df_driver_rank_stats.select("positionOrder", "code", "avg_pit_time").show(20)

+-------------+----+------------------+
|positionOrder|code|      avg_pit_time|
+-------------+----+------------------+
|            1| ALO| 21.06982354556813|
|            1| PIA| 21.21500015258789|
|            1| MAL|21.222333908081055|
|            1| RUS|21.991666793823242|
|            1| BUT|22.091470325694363|
|            1| RAI|22.220499992370605|
|            1| VET|22.751891302025836|
|            1| BOT| 23.10953844510592|
|            1| WEB|           23.1875|
|            1| NOR|23.485999584197998|
|            1| ROS| 69.99572730064392|
|            1| RIC|   98.553882374483|
|            1| HAM| 98.72578315849763|
|            1| VER|131.12980795192718|
|            1| LEC| 217.4418331782023|
|            1| PER|306.90900004993784|
|            1| SAI|453.69257164001465|
|            1| OCO| 503.7846663792928|
|            1| GAS| 771.9875001907349|
|            2| KVY|19.922200012207032|
+-------------+----+------------------+
only showing top 20 rows


To incorporate driver abbreviation codes into the analysis results, I employed the following logic:
Introduction of a Dimension Table: I loaded the `drivers.csv` dataset. This table serves as a "dictionary" for driver information, containing each `driverId` along with its corresponding code, name, and nationality. Building upon the existing merged table—which combines pit stop and race result data—I joined it once again with the `drivers` table using the `driverId`. The core logic here is that the `driverId` acts as the bridge connecting behavioral data (pit stops) with attribute data (driver codes).
From this massive, merged table, I selected only the columns required for our analysis: `raceId`, `positionOrder`, `code`, and the converted `duration_float`.
Finally, I grouped the data by `code` and `positionOrder` to calculate the average pit stop duration. This approach allows us to view not only the rankings but also the specific performance associated with each individual driver's abbreviation code.

EXplain
`df_drivers = spark.read...` — Loads the table containing static information about the drivers. Setting `inferSchema=True` is crucial here, as it automatically identifies `driverId` as an integer, thereby facilitating subsequent join operations.

`.join(df_drivers, "driverId")` — The second join operation. This step uses `driverId` to "insert" the driver's abbreviation code into every row of the pit stop records.

`.groupBy("positionOrder", "code")` — A two-dimensional grouping operation. This allows us to observe the pit stop performance of specific drivers (identified by their `code`) at specific finishing positions (`positionOrder`).

`df_full_merged` — This embodies the concept of a "wide table." In data engineering, consolidating information scattered across various CSV files into a single wide table is a prerequisite for conducting in-depth analysis.

In [0]:
#4.  Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

from pyspark.sql import functions as F

# 1. 
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True, inferSchema=True)

# 2. 
df_age_base = df_results.select("raceId", "driverId") \
    .join(df_races.select("raceId", "name", "date"), "raceId") \
    .join(df_drivers.select("driverId", "forename", "surname", "dob"), "driverId")

df_with_age = df_age_base.withColumn("Age", 
    F.floor(F.datediff(F.col("date"), F.col("dob")) / 365)
)

# 3. 
df_race_ages = df_with_age.groupBy("raceId", "name") \
    .agg(
        F.min("Age").alias("Youngest_Age"),
        F.max("Age").alias("Oldest_Age")
    ) \
    .orderBy("raceId")

# Result
df_race_ages.show(10)

+------+--------------------+------------+----------+
|raceId|                name|Youngest_Age|Oldest_Age|
+------+--------------------+------------+----------+
|     1|Australian Grand ...|          20|        36|
|     2|Malaysian Grand Prix|          20|        36|
|     3|  Chinese Grand Prix|          20|        36|
|     4|  Bahrain Grand Prix|          20|        36|
|     5|  Spanish Grand Prix|          20|        36|
|     6|   Monaco Grand Prix|          20|        37|
|     7|  Turkish Grand Prix|          20|        37|
|     8|  British Grand Prix|          20|        37|
|     9|   German Grand Prix|          20|        37|
|    10|Hungarian Grand Prix|          19|        37|
+------+--------------------+------------+----------+
only showing top 10 rows


To identify the youngest and oldest drivers in each race, we require the following tables:
`pit_stops` (or `results`): To determine which drivers participated in which races.
`races`: To retrieve the specific date of each race.
`drivers`: To retrieve the birth dates (DOB) of the drivers.

Defining "Age":
My definition: Age refers to a driver's actual age (in full years) on the day of the race.
Calculation Logic: Use `DATEDIFF` to calculate the number of days between the race date and the driver's birth date, then divide by 365 to obtain the precise age in years.

Calculating Extremes:

Group the data by `raceId`.
Within each group, identify the minimum (`MIN`) and maximum (`MAX`) values ​​for Age (representing the youngest and oldest drivers, respectively).
Linking Identities: To identify *who* these extreme values ​​correspond to, we must either join the results back to the driver information table or use window functions to extract the details directly.

EXplain
F.datediff(end, start)	Calculates the difference in days between two dates. In the context of F1 historical analysis, this is a more precise method than `year(date) - year(dob)`, as it takes specific months and days into account.

F.floor(...)	Rounds down to the nearest integer. For example, if a driver is 19.9 years old, in both legal and sports statistical contexts, they are still considered to be 19 years old until their actual birthday arrives.

365	Used to convert a duration expressed in days into years.

F.min("Age") / F.max("Age")	Aggregate functions. In the context of `groupBy("raceId")`, these functions respectively retrieve the minimum and maximum age values ​​among all competitors participating in that specific race.

In [0]:
#At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. 
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)

# 2.
df_combined = df_results.join(df_races.select("raceId", "year", "round"), "raceId")

# 3. 
df_flags = df_combined.withColumn("is_win", F.when(F.col("positionOrder") == 1, 1).otherwise(0)) \
                      .withColumn("is_2nd", F.when(F.col("positionOrder") == 2, 1).otherwise(0)) \
                      .withColumn("is_3rd", F.when(F.col("positionOrder") == 3, 1).otherwise(0))

# 4. Group by rider, sort by time.
# The range extends from the start to the current row (unboundedPreceding to currentRow).
window_history = Window.partitionBy("driverId").orderBy("year", "round").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 5. 
df_final = df_flags.withColumn("total_wins", F.sum("is_win").over(window_history)) \
                   .withColumn("total_2nd", F.sum("is_2nd").over(window_history)) \
                   .withColumn("total_3rd", F.sum("is_3rd").over(window_history))

# RUselt
df_final.select("year", "round", "driverId", "positionOrder", "total_wins", "total_2nd", "total_3rd") \
        .filter(F.col("driverId") == 1) \
        .orderBy("year", "round").show(10)

+----+-----+--------+-------------+----------+---------+---------+
|year|round|driverId|positionOrder|total_wins|total_2nd|total_3rd|
+----+-----+--------+-------------+----------+---------+---------+
|2007|    1|       1|            3|         0|        0|        1|
|2007|    2|       1|            2|         0|        1|        1|
|2007|    3|       1|            2|         0|        2|        1|
|2007|    4|       1|            2|         0|        3|        1|
|2007|    5|       1|            2|         0|        4|        1|
|2007|    6|       1|            1|         1|        4|        1|
|2007|    7|       1|            1|         2|        4|        1|
|2007|    8|       1|            3|         2|        4|        2|
|2007|    9|       1|            3|         2|        4|        3|
|2007|   10|       1|            9|         2|        4|        3|
+----+-----+--------+-------------+----------+---------+---------+
only showing top 10 rows


To calculate the total number of podium finishes each driver has accumulated at any specific point in time during a race:
Data Flagging: Create three indicator columns (1st, 2nd, 3rd) within the `results` table. If `positionOrder` equals 1, the 'win' column is assigned a value of 1; otherwise, it is 0. Apply the same logic for 2nd and 3rd place finishes.
Window Definition: We require a window partitioned by 'driver' and ordered in ascending sequence by 'date' (or 'season/round').
Running Total: Utilize window functions to calculate the cumulative count of wins (or podiums) from the very first race of that driver's career up to—but not including—the current race.

My interpretation is that this metric represents the cumulative data *including* the current race—or, more broadly, the achievements attained *up to* the point of this specific race. Therefore, I have opted to use the cumulative value that *includes* the current race results.
Linking Race Information: Join with the `races` table to ensure that the cumulative calculations are performed in strict chronological order (by year and round), rather than merely by `raceId`.

Explain
Window.partitionBy("driverId")	Core Logic: Ensures that cumulative calculations are performed exclusively within the records of a single driver, preventing aggregation across different drivers.

orderBy("year", "round")	Establishes the Timeline: Guarantees that cumulative calculations proceed according to the chronological order of the seasons, reflecting the historical progression of events.

rowsBetween(...)	Defines the Calculation Window: `unboundedPreceding` refers to the driver's very first race, while `currentRow` refers to the current race being processed. This implements a "cumulative-to-date" aggregation.

F.sum(...).over(window)	Performs a windowed summation on the indicator column (containing values ​​of 0 or 1) to derive the cumulative count.

F.when(...).otherwise(0)	Converts a categorical variable (the driver's rank) into a numerical variable (0 or 1), thereby enabling subsequent summation operations.

In [0]:
# How significant is the impact of grid position on the final finishing order?


My Logic:
Extract `grid` (starting position) and `positionOrder` (finishing position) from the `results` table. Since some drivers start from the Pit Lane or have anomalous starting position data, we must filter out invalid entries where `grid = 0`.
Calculate "Position Change" (Position Gain/Loss):
Create a new column, `position_change = grid - positionOrder`.
If the result is positive, it indicates the driver overtook others during the race; if negative, it indicates a drop in position.

Conversion Rate as Probability: Calculate the percentage of instances where a specific starting position (e.g., starting in 1st place) ultimately results in winning the championship (finishing in 1st place).
Aggregate Analysis: Group the data by `grid` to calculate the average finishing position.

In [0]:
from pyspark.sql import functions as F

# 1. 
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)

# 2.
df_valid_grid = df_results.filter(F.col("grid") > 0)

# 3. 
df_grid_analysis = df_valid_grid.groupBy("grid") \
    .agg(
        F.avg("positionOrder").alias("avg_finish_pos"),
        F.count("raceId").alias("total_starts"),
    
        F.sum(F.when(F.col("positionOrder") == 1, 1).otherwise(0)).alias("wins")
    )

# 4. 
df_grid_final = df_grid_analysis.withColumn("win_rate_percent", 
    F.round((F.col("wins") / F.col("total_starts")) * 100, 2)
)

# Result
df_grid_final.orderBy("grid").select("grid", "avg_finish_pos", "win_rate_percent").show(10)

+----+------------------+----------------+
|grid|    avg_finish_pos|win_rate_percent|
+----+------------------+----------------+
|   1| 5.663732394366197|           42.34|
|   2| 6.611555555555555|           23.82|
|   3| 7.505309734513275|           12.12|
|   4| 8.231448763250883|            6.01|
|   5| 9.136042402826854|            4.33|
|   6| 9.574222222222222|            3.56|
|   7|10.066960352422907|            2.03|
|   8|10.688219663418955|            1.51|
|   9|10.803886925795053|            0.44|
|  10|11.712389380530974|            1.06|
+----+------------------+----------------+
only showing top 10 rows


F.filter(F.col("grid") > 0) — Excludes anomalous data. In raw F1 data, a value of 0 or a special marker typically indicates a start from the pit lane, which does not constitute a standard grid starting position.

F.avg("positionOrder") — Measures the "baseline potential" of a given starting position. For instance, if the average finishing position for a driver starting in 1st place is 2.5, this suggests that starting from pole position offers a significant advantage.

F.sum(F.when(...)) — Conditional summation/counting. Used to tally how many drivers, starting from a specific grid position, ultimately reached the top step of the podium.

F.round(..., 2) — Rounds complex floating-point win probabilities to two decimal places, ensuring the data report aligns with professional business presentation standards.